# Qwen-Image-2.1-Turbo 在 Google Colab 上运行

基于 [abenzerps/Qwen-Image-2.1-Turbo-Quantized](https://huggingface.co/abenzerps/Qwen-Image-2.1-Turbo-Quantized)，
在 Colab 免费 T4 (16GB VRAM) 上部署，并提供 **OpenAI 风格 `/v1/images/generations` 接口**（端口 `8010`），
再用 `cloudflared` 暴露公网地址。

## 模型选型（Colab 推荐）

| 量化文件 | 大小 | 建议 |
|---|---|---|
| `qwen-image-2.1-turbo-Q4_K_M.gguf` | 4.05 GB | ✅ Colab 首选，体积与质量平衡 |
| `qwen-image-2.1-turbo-Q4_0.gguf` | 4.05 GB | 显存极紧时的备选 |
| `qwen-image-2.1-turbo-Q5_K_M.gguf` | 4.94 GB | 质量更好，稍慢，T4 16GB 可跑 |
| `qwen-image-2.1-turbo-Q6_K.gguf` | 5.88 GB | 显存吃紧，慎选 |
| `qwen-image-2.1-turbo-Q8_0.gguf` | 7.59 GB | 一般不建议在免费 Colab 上使用 |

必须配套下载（工作流三件套）：
- 文本编码器：`text_encoders/qwen3vl_8b_int8_convrot.safetensors` (9.35 GB，INT8 更省显存)
- VAE：`vae/qwen_image_2.1_vae_bf16.safetensors` (0.68 GB)

合计约 **14 GB** 下载；Colab 免费版磁盘通常充足，下载需 10–30 分钟，视网络而定。

> **注意**：这是文生图（DiT 7B）模型，**不适用 `llama-cpp-python`**（那是文本模型推理库）。
本笔记本采用 ComfyUI + [ComfyUI-GGUF](https://github.com/leejet/ComfyUI-GGUF) 运行 GGUF 权重。

**关于 FastAPI 这层**：ComfyUI 自带完整 HTTP API（`/prompt`、`/history`、`/view`），本身就能出图，
FastAPI **不是必需的**，它只是把 ComfyUI 的工作流 JSON 包装成 OpenAI 兼容的
`/v1/images/generations` + `b64_json`，方便沿用 OpenAI 客户端。
若不想要这层，把隧道 Cell 里的 `TUNNEL_PORT` 改成 `8188`，直接访问 ComfyUI 原生 API 即可。

运行前：`Runtime → Change runtime type → T4 GPU`，保存后重新运行全部 Cell。


In [ ]:
import sys, os
print('Python:', sys.version)
try:
    import torch
    print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
except Exception as e:
    print('torch 未安装:', e)
!nvidia-smi
!df -h /content | tail -1


In [ ]:
import urllib.request, time
MIRRORS = [
    ("huggingface.co", "https://huggingface.co"),
    ("hf-mirror.com", "https://hf-mirror.com"),
    ("gitcode.com/hf_mirrors", "https://gitcode.com/hf_mirrors"),
]
TEST_PATH = "/abenzerps/Qwen-Image-2.1-Turbo-Quantized/resolve/main/qwen-image-2.1-turbo-Q4_K_M.gguf"

def probe(base):
    req = urllib.request.Request(base + TEST_PATH, method="HEAD")
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            return round(time.time() - t0, 2), int(r.headers.get('Content-Length', 0)), r.status
    except Exception as e:
        return None, 0, repr(e)[:80]

rows = []
for name, base in MIRRORS:
    dt, size, status = probe(base)
    rows.append((name, base, dt, size, status))
    print(f'{name:26s} {str(dt):>6}s  status={status}  size={size/1e9:.2f} GB')

ok = [r for r in rows if r[2] is not None]
BASE = min(ok, key=lambda r: r[2])[1] if ok else "https://huggingface.co"
print(">>> 选用镜像源:", BASE)


In [ ]:
import os, subprocess, sys
os.chdir('/content')
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
os.chdir('/content/ComfyUI')
!pip install -r requirements.txt -q -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install -q fastapi uvicorn requests -i https://pypi.tuna.tsinghua.edu.cn/simple
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/leejet/ComfyUI-GGUF.git custom_nodes/ComfyUI-GGUF
print('ComfyUI 就绪')


In [ ]:
import os
MODEL_DIR = '/content/ComfyUI/models'
FILES = [
    ('diffusion_models/qwen-image-2.1-turbo-Q4_K_M.gguf', 'qwen-image-2.1-turbo-Q4_K_M.gguf', 4.05e9),
    ('text_encoders/qwen3vl_8b_int8_convrot.safetensors', 'qwen3vl_8b_int8_convrot.safetensors', 9.35e9),
    ('vae/qwen_image_2.1_vae_bf16.safetensors', 'qwen_image_2.1_vae_bf16.safetensors', 0.68e9),
]
for rel, fname, expect in FILES:
    dst = os.path.join(MODEL_DIR, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) and abs(os.path.getsize(dst) - expect) < expect * 0.05:
        print('已存在，跳过:', rel)
        continue
    url = BASE + '/abenzerps/Qwen-Image-2.1-Turbo-Quantized/resolve/main/' + fname
    print('下载:', url)
    !wget -c -q --show-progress "$url" -O "$dst"
    got = os.path.getsize(dst)
    print('完成:', rel, '%.2f GB' % (got / 1e9))


In [ ]:
import subprocess, sys, time, socket, os

def port_open(p, host='127.0.0.1'):
    s = socket.socket(); s.settimeout(1)
    try:
        s.connect((host, p)); return True
    except Exception:
        return False
    finally:
        s.close()

if port_open(8188):
    print('ComfyUI 已在运行: http://127.0.0.1:8188')
else:
    log = open('/content/comfyui.log', 'a')
    comfy = subprocess.Popen([sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
                             cwd='/content/ComfyUI', stdout=log, stderr=log, start_new_session=True)
    print('ComfyUI PID:', comfy.pid)
    for _ in range(60):
        time.sleep(5)
        if port_open(8188):
            break
    if port_open(8188):
        print('ComfyUI 已就绪: http://127.0.0.1:8188')
    else:
        print('❌ 启动失败，日志末尾 30 行:' )
        print('\n'.join(open('/content/comfyui.log').read().splitlines()[-30:]))


In [ ]:
%%writefile /content/qwen_api.py
# OpenAI 兼容代理：把 ComfyUI 工作流包装成 /v1/images/generations
# 若 ComfyUI-GGUF 节点类名不同，改 UNET_CLASS 即可
from fastapi import FastAPI
from pydantic import BaseModel
import requests, base64, uuid, time, uvicorn, traceback

app = FastAPI(title='Qwen-Image-2.1-Turbo proxy')
COMFY = 'http://127.0.0.1:8188'
UNET = 'qwen-image-2.1-turbo-Q4_K_M.gguf'
CLIP = 'qwen3vl_8b_int8_convrot.safetensors'
VAE = 'qwen_image_2.1_vae_bf16.safetensors'
UNET_CLASS = 'UNETLoaderGGUF'

class ImageGenRequest(BaseModel):
    prompt: str
    model: str = UNET
    width: int = 1024
    height: int = 1024
    steps: int = 8
    seed: int = -1
    cfg: float = 1.0

@app.get('/')
def root():
    return {'status': 'ok', 'backend': COMFY, 'endpoint': '/v1/images/generations'}

@app.get('/health')
def health():
    try:
        requests.get(COMFY + '/system_stats', timeout=5).raise_for_status()
        return {'ok': True, 'comfyui': 'up'}
    except Exception as e:
        return {'ok': False, 'comfyui': repr(e)}

@app.get('/v1/models')
def models():
    return {'object': 'list', 'data': [{'id': UNET, 'object': 'model'}]}

def build_workflow(req: ImageGenRequest):
    seed = req.seed if req.seed and req.seed >= 0 else uuid.uuid4().int >> 64
    return {
        '1': {'inputs': {'unet_name': req.model}, 'class_type': UNET_CLASS},
        '2': {'inputs': {'clip_name': CLIP, 'type': 'qwen_image'}, 'class_type': 'CLIPLoader'},
        '3': {'inputs': {'vae_name': VAE}, 'class_type': 'VAELoader'},
        '4': {'inputs': {'width': req.width, 'height': req.height, 'batch_size': 1}, 'class_type': 'EmptyLatentImage'},
        '5': {'inputs': {'model': ['1', 0], 'positive': ['2', 0], 'negative': ['2', 0], 'latent_image': ['4', 0],
                'seed': int(seed), 'steps': req.steps, 'cfg': req.cfg,
                'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0}, 'class_type': 'KSampler'},
        '6': {'inputs': {'samples': ['5', 0], 'vae': ['3', 0]}, 'class_type': 'VAEDecode'},
        '7': {'inputs': {'images': ['6', 0], 'filename_prefix': 'qwen_colab'}, 'class_type': 'SaveImage'},
    }

@app.post('/v1/images/generations')
def generate(req: ImageGenRequest):
    try:
        r = requests.post(COMFY + '/prompt', json={'prompt': build_workflow(req)}, timeout=30)
        if r.status_code != 200:
            return {'error': 'comfyui /prompt rejected', 'detail': r.text[:1000]}
        pid = r.json()['prompt_id']
        outs = None
        for _ in range(180):
            time.sleep(2)
            h = requests.get(COMFY + f'/history/{pid}', timeout=10).json()
            if pid in h and 'outputs' in h[pid]:
                outs = h[pid]['outputs']
                break
        if outs is None:
            return {'error': 'generation timeout'}
        images = []
        for node_id, out in outs.items():
            for img in out.get('images', []):
                data = requests.get(COMFY + '/view',
                    params={'filename': img['filename'], 'subfolder': img.get('subfolder', ''), 'type': 'output'},
                    timeout=60).content
                images.append({'b64_json': base64.b64encode(data).decode('ascii')})
        return {'created': int(time.time()), 'data': images}
    except Exception:
        return {'error': 'proxy exception', 'detail': traceback.format_exc()[-1500:]}

if __name__ == '__main__':
    print('qwen_api: listening on 0.0.0.0:8010 -> ' + COMFY, flush=True)
    uvicorn.run(app, host='0.0.0.0', port=8010, log_level='info')


In [ ]:
import subprocess, sys, time, os

if not os.path.exists('/content/qwen_api.py'):
    raise RuntimeError('缺少 /content/qwen_api.py —— 请先运行上一单元格（%%writefile）')

# 清掉可能残留的死进程，避免端口占用
subprocess.run("ps -eo pid,args | grep '[q]wen_api.py' | awk '{print $1}' | xargs -r kill", shell=True)
time.sleep(1)
api = subprocess.Popen([sys.executable, 'qwen_api.py'], cwd='/content',
                       stdout=open('/content/qwen_api.log', 'w'), stderr=subprocess.STDOUT,
                       start_new_session=True)
print('FastAPI 代理 PID:', api.pid)
for _ in range(40):
    if port_open(8010):
        break
    time.sleep(2)
if port_open(8010):
    print('✅ FastAPI 已就绪: http://127.0.0.1:8010/v1/images/generations')
else:
    print('❌ 8010 未监听，/content/qwen_api.log 末尾:' )
    print(open('/content/qwen_api.log').read()[-2000:])


In [ ]:
import subprocess, re, time, os, urllib.request

TUNNEL_PORT = 8010   # 改成 8188 即直连 ComfyUI 原生 API（不需要 FastAPI 这层）

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

running = subprocess.run("ps -eo args | grep -c '[c]loudflared tunnel'", shell=True,
                         capture_output=True, text=True).stdout.strip()
if running != '0':
    print('已有 cloudflared 隧道进程，保持原公网域名不变；后端恢复后自动解除 502')
else:
    tlog = open('/content/cloudflared.log', 'w')
    subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{TUNNEL_PORT}'],
                     stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
    public = None
    for _ in range(40):
        time.sleep(2)
        txt = open('/content/cloudflared.log').read()
        m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', txt)
        if m:
            public = m[0]
            break
    print('>>> 公网地址:', public)
    if TUNNEL_PORT == 8010 and public:
        print('>>> OpenAI 兼容接口:', public + '/v1/images/generations')
    elif public:
        print('>>> ComfyUI 原生 API:', public + '/system_stats')


In [ ]:
import requests, base64, time
from IPython.display import Image, display
payload = {
    'prompt': '一只穿宇航服的猫坐在月球表面，背景是地球升起，电影级光照，超细节，4K',
    'width': 1024, 'height': 1024, 'steps': 8, 'seed': 42, 'cfg': 1.0
}
t0 = time.time()
r = requests.post('http://127.0.0.1:8010/v1/images/generations', json=payload, timeout=600)
dt = time.time() - t0
print('HTTP', r.status_code, '| 端到端耗时: %.1f 秒' % dt)
res = r.json()
if 'error' in res:
    print('❌', res.get('error'), '\n', res.get('detail', '')[:1500])
else:
    img_b64 = res['data'][0]['b64_json']
    open('/content/qwen_image_test.png', 'wb').write(base64.b64decode(img_b64))
    display(Image(filename='/content/qwen_image_test.png'))


# 用你自己的 OpenAI 客户端调用

```python
from openai import OpenAI
client = OpenAI(base_url='https://<你的 cloudflared 域名>/v1', api_key='not-needed')
resp = client.images.generate(
    model='qwen-image-2.1-turbo-Q4_K_M.gguf',
    prompt='春日樱花下的江南小镇，水墨风格，8K',
    width=1024, height=1024
)
img_b64 = resp.data[0].b64_json
```

### FAQ
- **跑不起来？** 确认 Colab 运行时选了 **T4 GPU**；本地 CPU 推理 DiT 7B 太慢，不建议。
- **显存不足？** 把 `qwen3vl_8b_int8_convrot.safetensors` 换更小的量化，或把 `width/height` 降到 768。
- **下载慢？** 镜像探测 Cell 会自动选最快源；pip 走清华源。
- **公网域名 502？** 隧道活着但 8010 没监听。按顺序重跑「写入 qwen_api.py → 启动代理」两个 Cell；
  用 `curl http://127.0.0.1:8010/health` 确认后端，再刷新公网域名。隧道进程保留则域名不变。
- **不想要 FastAPI？** 把隧道 Cell 的 `TUNNEL_PORT` 改成 `8188`，直接用 ComfyUI 原生 `/prompt`、`/history`、`/view`。
- **接口风格：** 保留 OpenAI `images.generate` 兼容；`b64_json` 直接解码为图片。
- **测速指标：** 测试 Cell 打印单图端到端耗时（含 ComfyUI 排队与去噪），可用于对比不同量化版本。
